In [11]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../..')
import data_preprocessing.coil20 as coil20
from importlib import reload
reload(coil20)

<module 'data_preprocessing.coil20' from '/Users/apple/Library/Mobile Documents/3L68KQB4HG~com~readdle~CommonDocuments/Documents/本学期资料/MSBD5002Project/implementation/experiment/1_regularization_term/../../data_preprocessing/coil20.py'>

In [33]:
mat, label = coil20.read_data()
mat = torch.tensor(mat, dtype=torch.float32)

A = mat @ mat.T

In [38]:
def loss_without_regularization(A, H, lambda_=None):
    '''
    A: a symmetric matrix of size n x n
    H: a matrix of size n x k
    '''
    assert A.T.equal(A), "A is not symmetric"
    
    return torch.norm(A - H @ H.T) ** 2

def loss_with_regularization_optimized(A, H, lambda_):
    '''
    A: a symmetric matrix of size n x n
    H: a matrix of size n x k
    lambda_: a scalar
    '''
    assert A.T.equal(A), "A is not symmetric"
    
    loss = torch.norm(A - H @ H.T) ** 2
    
    # Vectorized computation of regularization loss
    diff = H[:, None, :] - H[None, :, :]
    regularization_loss = (A.unsqueeze(-1) * diff**2).sum()
    regularization_loss *= lambda_
    
    return loss + regularization_loss

In [52]:
def Gradient_Descent(A, H, lamda, 
                     loss_fn,
                     lr,
                     max_iter=100, tol=1e-4, print_every=100):
    '''
    A: a symmetric matrix of size n x n
    H: a matrix of size n x k
    lambda_: a scalar
    '''
    assert H.requires_grad, "H is not set to require grad"
    if H.grad is not None:
        H.grad.zero_()
    for i in range(max_iter):
        loss = loss_fn(A, H, lamda)
        loss.backward()
        with torch.no_grad():
            H -= lr * H.grad
            H.grad.zero_()
        if i % print_every == 0:
            print(f'iteration {i}, loss {loss.item()}')
        if loss < tol:
            break
    return H, loss

In [56]:
N = A.shape[0]
k = 8
H = torch.randn(N, k)
H.requires_grad = True

In [ ]:
rst_1 = Gradient_Descent(A=A, H=H, lamda=0, 
                         loss_fn=loss_without_regularization, 
                         lr=1e-6, max_iter=1000,
                         tol=1e-4, print_every=10)